In [ ]:
!pip install ultralytics

In [ ]:
import zipfile
import os

zip_path = '/content/agent_rag - Kopya (2).zip'
extract_path = '/content/dataset'

if not os.path.exists(extract_path):
    os.makedirs(extract_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Zip file extracted successfully! Current directory structure:")
!ls -R /content/dataset

In [ ]:
import os
from ultralytics import YOLO

yaml_content = """
path: /content/dataset/agent_rag - Kopya (2)
train: train/images
val: val/images
test: test/images

names:
  0: helmet_on
  1: helmet_off
  2: harness_on
  3: harness_off
"""

with open('/content/data.yaml', 'w') as f:
    f.write(yaml_content)

model = YOLO('yolo11n.pt')

results = model.train(
    data='/content/data.yaml',
    epochs=100,
    imgsz=640,
    patience=15,
    save=True,
    project='isg_projesi',
    name='baret_kemer_modeli',
    verbose=True
)

print("\n" + "="*30)
print("TRAINING COMPLETED. TEST RESULTS:")
metrics = model.val()
print(f"mAP50: {metrics.box.map50:.4f}")

In [ ]:
import os
import glob
import cv2
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

try:
    model_path = glob.glob('/content/runs/detect/*/weights/best.pt')[-1]
    print(f"Model Path: {model_path}")
    model = YOLO(model_path)
except:
    print("Model file not found. Check the 'runs' directory.")

test_img_path = '/content/dataset/agent_rag - Kopya (2)/test/images/ChatGPT Image Mar 7, 2026, 10_53_55 PM.png'

if os.path.exists(test_img_path):
    results = model.predict(source=test_img_path, conf=0.25)

    for r in results:
        res_plotted = r.plot()
        print(f"\n📸 Analyzed: {os.path.basename(test_img_path)}")
        cv2_imshow(res_plotted)

        for box in r.boxes:
            c = int(box.cls)
            print(f"-> Detection: {model.names[c]} (Confidence: {float(box.conf):.2f})")
else:
    print("Image file not found at the specified path.")

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Find all results.png files in the workspace
result_files = glob.glob('/content/**/results.png', recursive=True)
confusion_files = glob.glob('/content/**/confusion_matrix.png', recursive=True)

if not result_files:
    print("❌ ERROR: results.png not found.")
    print("Possible reasons: 1. Training is still running. 2. Training crashed. 3. Results were not saved.")
else:
    # Pick the most recent results file
    latest_results = max(result_files, key=os.path.getctime)
    print(f"✅ Found Results: {latest_results}")

    # Plot Training Metrics
    plt.figure(figsize=(15, 10))
    img = mpimg.imread(latest_results)
    plt.imshow(img)
    plt.axis('off')
    plt.title("TRAINING METRICS (Loss, mAP, Precision, Recall)", fontsize=18)
    plt.show()

# Show Confusion Matrix if available
if confusion_files:
    latest_confusion = max(confusion_files, key=os.path.getctime)
    print(f"✅ Found Confusion Matrix: {latest_confusion}")

    plt.figure(figsize=(12, 10))
    img = mpimg.imread(latest_confusion)
    plt.imshow(img)
    plt.axis('off')
    plt.title("CONFUSION MATRIX", fontsize=15)
    plt.show()
else:
    print("❌ Confusion Matrix not found yet.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os

# Find ANY results.csv in the content directory
csv_files = glob.glob('/content/**/results.csv', recursive=True)

if not csv_files:
    print("❌ ERROR: results.csv not found anywhere. Please check if the training actually started.")
else:
    # Get the latest one
    latest_csv = max(csv_files, key=os.path.getctime)
    print(f"📊 Found it! Plotting from: {latest_csv}")

    df = pd.read_csv(latest_csv)
    df.columns = df.columns.str.strip() # Clean column names

    # Create 2x2 Clean Grid
    fig, axs = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('CORE PERFORMANCE METRICS', fontsize=22, fontweight='bold')

    # 1. BOX LOSS (Lower is better)
    axs[0, 0].plot(df['train/box_loss'], color='#e74c3c', linewidth=2)
    axs[0, 0].set_title('Localization Error (Box Loss)', fontsize=14)
    axs[0, 0].grid(True, alpha=0.3)

    # 2. CLASS LOSS (Lower is better)
    axs[0, 1].plot(df['train/cls_loss'], color='#f39c12', linewidth=2)
    axs[0, 1].set_title('Classification Error (Class Loss)', fontsize=14)
    axs[0, 1].grid(True, alpha=0.3)

    # 3. PRECISION (Higher is better)
    # Note: Column names might vary slightly in YOLO versions, using direct index if needed
    p_col = 'metrics/precision(B)' if 'metrics/precision(B)' in df.columns else df.columns[10]
    axs[1, 0].plot(df[p_col], color='#2980b9', linewidth=2)
    axs[1, 0].set_title('Precision (Accuracy)', fontsize=14)
    axs[1, 0].grid(True, alpha=0.3)

    # 4. mAP50 (The Final Grade - Higher is better)
    map_col = 'metrics/mAP50(B)' if 'metrics/mAP50(B)' in df.columns else df.columns[12]
    axs[1, 1].plot(df[map_col], color='#27ae60', linewidth=2)
    axs[1, 1].set_title('Overall Success (mAP50)', fontsize=14)
    axs[1, 1].grid(True, alpha=0.3)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

In [ ]:
import os
import glob
from google.colab import files

find_model = glob.glob('**/best.pt', recursive=True)

if find_model:
    latest_model = max(find_model, key=os.path.getctime)
    print(f"✅ Found the model at: {latest_model}")
    print("🚀 Starting download... Please wait for the browser popup.")
    files.download(latest_model)
else:
    print("❌ Still can't find best.pt.")
    manual_path = '/content/runs/detect/isg_project/ppe_detection_model/weights/best.pt'
    if os.path.exists(manual_path):
        print(f"✅ Manual path worked: {manual_path}")
        files.download(manual_path)
    else:
        print("🔍 Listing all files in runs to see what's happening:")
        !find /content/runs -name "*.pt"

In [ ]:
from ultralytics import YOLO
import cv2
from google.colab.patches import cv2_imshow
import os

model_path = '/content/best.pt'
if os.path.exists(model_path):
    model = YOLO(model_path)
    print("✅ Model successfully loaded.")
else:
    print("❌ Model file not found at /content/best.pt! Please upload it first.")

img_path = '/content/Gemini_Generated_Image_7uth3w7uth3w7uth.png'

if os.path.exists(img_path):
    results = model.predict(source=img_path, conf=0.25)

    for r in results:
        res_plotted = r.plot()

        print(f"\n📸 Analysis Result for: {os.path.basename(img_path)}")
        cv2_imshow(res_plotted)

        print("\n--- Detection Details ---")
        for box in r.boxes:
            c = int(box.cls)
            label = model.names[c]
            confidence = float(box.conf)
            print(f"-> Found: {label} | Confidence: {confidence:.2f}")
else:
    print(f"❌ Image not found at: {img_path}")